In [2]:
import h5py
import numpy as np
import os

def get_dataset_name(file_name_with_dir):
    filename_without_dir = os.path.basename(file_name_with_dir)
    name_no_ext, _ = os.path.splitext(filename_without_dir)
    dataset_name = '_'.join(name_no_ext.split('_')[:-1])
    return dataset_name

In [3]:
def read_h5_file(filename_path):
    dataset_name = get_dataset_name(filename_path)

    with h5py.File(filename_path, "r") as f:
        matrix = f.get(dataset_name)[()]

    return matrix

## Normalizing using zscore

In [4]:
def zscore_normalize(matrix):
    mean = matrix.mean(axis=1, keepdims=True)
    std = matrix.std(axis=1, keepdims=True)

    normalized_matrix = (matrix - mean) / (std + 1e-6)

    return normalized_matrix

## Downsampling using average pool

In [5]:
def average_pool_downsample(matrix, factor=10):
    n_channels, n_timepoints = matrix.shape

    usable_length = (n_timepoints // factor) * factor
    matrix = matrix[:, :usable_length]

    matrix = matrix.reshape(n_channels, usable_length // factor, factor)

    downsampled_matrix = matrix.mean(axis=2)

    return downsampled_matrix

In [6]:
label_map = {
    "rest": 0,
    "task_story_math": 1,
    "task_working_memory": 2,
    "task_motor": 3
}

def extract_label(filename_path):
    filename = os.path.basename(filename_path)

    if filename.startswith("rest"):
        return label_map["rest"]
    elif filename.startswith("task_story_math"):
        return label_map["task_story_math"]
    elif filename.startswith("task_working_memory"):
        return label_map["task_working_memory"]
    elif filename.startswith("task_motor"):
        return label_map["task_motor"]
    else:
        raise ValueError(f"Unknown task type in file: {filename}")

In [22]:
def preprocess_file(filename_path, downsample_factor=1):
    matrix = read_h5_file(filename_path)
    matrix = average_pool_downsample(
        matrix,
        factor=downsample_factor
    )
    matrix = zscore_normalize(matrix)

    matrix = matrix.astype(np.float32)

    label = extract_label(filename_path)

    return matrix, label

In [ ]:
from collections import Counter


def load_and_preprocess_folder(folder_path, downsample_factor=1):
    X = []
    y = []
    file_names = []

    h5_files = [
        file for file in os.listdir(folder_path)
        if file.endswith(".h5")
    ]

    h5_files = sorted(h5_files)

    print(f"\nProcessing folder: {folder_path}")
    print(f"Number of h5 files: {len(h5_files)}")

    for file in h5_files:
        filename_path = os.path.join(folder_path, file)

        x_i, y_i = preprocess_file(
            filename_path,
            downsample_factor=downsample_factor
        )

        X.append(x_i)
        y.append(y_i)
        file_names.append(file)

    X = np.stack(X).astype(np.float32)
    y = np.array(y)

    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Label distribution:", Counter(y))

    return X, y, file_names

In [ ]:
base_dir = "Final Project data"
downsample_factor = 10

folders = {
    "intra_train": os.path.join(base_dir, "Intra", "train"),
    "intra_test": os.path.join(base_dir, "Intra", "test"),

    "cross_train": os.path.join(base_dir, "Cross", "train"),
    "cross_test1": os.path.join(base_dir, "Cross", "test1"),
    "cross_test2": os.path.join(base_dir, "Cross", "test2"),
    "cross_test3": os.path.join(base_dir, "Cross", "test3"),
}


# Preprocessing
## Preprocessing Output

The preprocessed data is stored in the `datasets` dictionary, with one entry per subject (`intra_train`, `intra_test`, `cross_train`, `cross_test1`, `cross_test2`, `cross_test3`).

Each entry is a dict with three fields:

| Field | Type | Shape | Description |
|---|---|---|---|
| `X` | `np.ndarray` (float32) | `(N, 248, 3562)` | MEG signals: N recordings × 248 channels × 3562 time steps (downsampled by factor 10) |
| `y` | `np.ndarray` (int64) | `(N,)` | Task labels: `0=rest`, `1=story_math`, `2=working_memory`, `3=motor` |
| `file_names` | `list[str]` | length `N` | Original `.h5` filenames, aligned with `X` and `y` |

**Example usage:**

```python
X_train = datasets["cross_train"]["X"]   # (64, 248, 3562)
y_train = datasets["cross_train"]["y"]   # (64,)
```

In [ ]:
datasets = {}
for subject, folder_path in folders.items():
    X, y, file_names = load_and_preprocess_folder(
        folder_path,
        downsample_factor=downsample_factor
    )

    datasets[subject] = {
        # meg signal
        "X": X,
        # task type
        "y": y,
        "file_names": file_names
    }


Processing folder: C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data\Intra\train
Number of h5 files: 32
X shape: (32, 248, 3562)
y shape: (32,)
Label distribution: Counter({np.int64(0): 8, np.int64(3): 8, np.int64(1): 8, np.int64(2): 8})

Processing folder: C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data\Intra\test
Number of h5 files: 8
X shape: (8, 248, 3562)
y shape: (8,)
Label distribution: Counter({np.int64(0): 2, np.int64(3): 2, np.int64(1): 2, np.int64(2): 2})

Processing folder: C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data\Cross\train
Number of h5 files: 64
X shape: (64, 248, 3562)
y shape: (64,)
Label distribution: Counter({np.int64(0): 16, np.int64(3): 16, np.int64(1): 16, np.int64(2): 16})

Processing folder: C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data\Cross\test1
Number of h5 files: 16
X shape: (16, 248, 3562)
y shape: (16,)
Label distribution: Counter({np.int64(0): 4, np.int64(3): 4, np.int64(1): 4, np.int64(2): 4})

Process